In [1]:
"""
stage_5.py -- Stage 5 (iteration 2): 5c2-raw hazard model, manifest-driven.

Frozen model contract (5c2-raw): grammar (bsince + ewm{2,6,18} per class, age, tod)
+ causal z-scored values @ {t-1, t-2, t-3}: signed & magnitude of every stream's
source column, leg amplitude, raw body signed & magnitude. Sign convention
confirming-positive (value * leg_dir).

Manifest-driven (iteration 2):
  - Fork axes (frame, session window, stream set) are READ from the Stage-0
    manifest, never redeclared here. Dropping a stream in Stage 0 (e.g. no-TICK)
    propagates automatically: grammar classes AND z-value channels both track the
    manifest stream set.
  - tod is derived from clock time (session_start), resolution-independent.
  - The manifest is baked into the model bundle so the booster carries its own
    contract; the worker asserts against it and prints it at startup.

Naming (iteration 2):
  SOURCE_PATH / src : the "source" oscillator file (HA OHLC + JMA + TICK + derivs),
                      Stage-0's input; rawer than bars/events. src is the primary
                      data frame, augmented in-place with raw body columns.
  RAW_FILE / raw1   : pure MNQ raw OHLC (transient; merged into src, then unused).
"""

import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score

In [2]:
# ---------------------------------------------------------------- CONFIG (per-run, explicit)
FRAME = 6
STAGE0_TAG = 'TICK-9-12am

MANIFEST_PATH = f"stage-0/stage0_manifest_{STAGE0_TAG}_{FRAME}s.json"
BARS_PATH = f"stage-0/bars_{STAGE0_TAG}_{FRAME}s.pqt"
EVENTS_PATH = f"stage-0/events_{STAGE0_TAG}_{FRAME}s.pqt"

ITER_DIR = "."                                   # iteration-2 root (encapsulated)
OUT_DIR = "stage-5"

VALID_FROM = "2025-07-01"
TRAIN_END = "2025-12-31"
TEST_FROM = "2026-01-01"
RESULT_TAG = "5raw_01-2025"

# frozen 5c2 architecture constants (NOT fork axes -- stay in code)
TAUS = (2.0, 6.0, 18.0)
VALUE_LAGS = (1, 2)                               # t-2, t-3 (t-1 shift is implicit)
ZWARM = 20
TOD_BIN_MIN = 30
BODY_OPEN_COL = "rawOpen"
BODY_CLOSE_COL = "rawLast"
BODY_TAG = "raw"

LGBM_PARAMS = dict(
    objective="binary", metric="binary_logloss", learning_rate=0.05,
    num_leaves=127, min_data_in_leaf=1000, feature_fraction=0.9,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    num_threads=16, verbosity=-1,
)
NUM_ROUNDS = 4000
EARLY_STOP = 200
STAGE5_ANCHOR = 0.27402
# ----------------------------------------------------------------

In [3]:
def load_manifest(path):
    with open(path) as f:
        man = json.load(f)
    start = pd.Timestamp(man["session_start"])
    end = pd.Timestamp(man["session_end"])
    man["_tod_start_min"] = start.hour * 60 + start.minute
    man["_session_min"] = (end.hour * 60 + end.minute) - man["_tod_start_min"]
    man["_n_tod"] = int(np.ceil(man["_session_min"] / TOD_BIN_MIN))
    man["_stream_names"] = [s["name"] for s in man["streams"]]              # non-self
    man["_stream_cols"] = [s["column"] for s in man["streams"]]             # source cols
    man["_self"] = man["self_stream"]
    return man

In [4]:
# ---------------------------------------------------------------- z primitives
def _expanding_z(x, warm):
    n = len(x)
    csum = np.concatenate(([0.0], np.cumsum(x)))
    csq = np.concatenate(([0.0], np.cumsum(x * x)))
    k = np.arange(n)
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = np.where(k > 0, csum[:-1] / np.maximum(k, 1), 0.0)
        var = np.where(k > 1, (csq[:-1] - k * mean * mean) / np.maximum(k - 1, 1), 0.0)
        std = np.sqrt(np.maximum(var, 0.0))
        z = np.where((k >= warm) & (std > 0), (x - mean) / std, 0.0)
    return z


def _welford_check(x, warm):
    z_vec = _expanding_z(x, warm)
    k, mean, M2 = 0, 0.0, 0.0
    z_ref = np.zeros(len(x))
    for t in range(len(x)):
        if k >= warm and M2 > 0:
            z_ref[t] = (x[t] - mean) / np.sqrt(M2 / (k - 1))
        k += 1
        d = x[t] - mean
        mean += d / k
        M2 += d * (x[t] - mean)
    return float(np.max(np.abs(z_vec - z_ref)))


# ---------------------------------------------------------------- Featurizer
def _bins_from_edges(edges):
    bins, lo = [], 1
    for e in edges:
        bins.append((lo, int(e)))
        lo = int(e) + 1
    return bins


def _age_bins_from_edges(edges):
    b = _bins_from_edges(edges)
    b.append((int(edges[-1]) + 1, 10 ** 9))
    return b

In [5]:
class Featurizer:
    def __init__(self, bars, events, manifest,
                 bin_edges=(1, 2, 3, 5, 8, 13, 21, 34),
                 age_edges=(1, 2, 3, 5, 8, 13, 21, 34, 55, 89)):
        self.manifest = manifest
        self.tod_start_min = manifest["_tod_start_min"]
        self.n_tod = manifest["_n_tod"]
        self.bin_edges = list(bin_edges)
        self.age_edges = list(age_edges)
        self.bins = _bins_from_edges(bin_edges)
        self.age_bins = _age_bins_from_edges(age_edges)

        # classes derived from manifest stream set (opp/conf per stream + SELF)
        self.classes = []
        for s in manifest["_stream_names"]:
            self.classes.append((s, "opp"))
            self.classes.append((s, "conf"))
        self.classes.append((manifest["_self"], "all"))

        self.grammar_names = []
        for (s, c) in self.classes:
            self.grammar_names.append(f"{s}|{c}|bsince")
            self.grammar_names += [f"{s}|{c}|ewm{t:g}" for t in TAUS]
        self.grammar_names += ["age", "tod"]

        ev = events[events["stream"].isin([c[0] for c in self.classes])]
        ev = ev[~((ev["stream"] != manifest["_self"]) & (ev["opposing"] == 0))]
        evg = dict(tuple(ev.groupby("date")))

        self.sessions = []
        bars = bars.sort_values("bar_index").reset_index(drop=True)
        for sess, g in bars.groupby("date", sort=True):
            n = len(g)
            first = int(g["bar_index"].iloc[0])
            tgt = g["is_target"].to_numpy()
            warm = g["warm"].to_numpy()
            ts = pd.DatetimeIndex(g["timestamp"])
            mins = ts.hour.to_numpy() * 60 + ts.minute.to_numpy() - self.tod_start_min
            tod = np.clip(mins // TOD_BIN_MIN, 0, self.n_tod - 1).astype(np.int16)
            lt_incl = np.maximum.accumulate(np.where(tgt, np.arange(n), -1))
            P = {}
            e = evg.get(sess)
            for (s, c) in self.classes:
                if e is None:
                    loc = np.empty(0, np.int64)
                else:
                    if s == manifest["_self"]:
                        sub = e[e["stream"] == s]
                    else:
                        want = 1 if c == "opp" else -1
                        sub = e[(e["stream"] == s) & (e["opposing"] == want)]
                    loc = sub["event_bar"].to_numpy() - first
                ind = np.zeros(n, np.int32)
                ind[loc] = 1
                P[(s, c)] = np.concatenate(([0], np.cumsum(ind)))
            self.sessions.append(dict(
                sess=sess, first=first, n=n, tgt=tgt, warm=warm, tod=tod,
                lt_incl=lt_incl, P=P,
                bar_index=g["bar_index"].to_numpy(),
                timestamp=g["timestamp"].to_numpy()))

    def _selected(self, date_from, date_to):
        for S in self.sessions:
            d = str(S["sess"])
            if date_from is not None and d < date_from:
                continue
            if date_to is not None and d > date_to:
                continue
            yield S


def augment_featurizer(fz, bars):
    """Attach leg_dir and leg_start JMA level per session (for value features)."""
    bg = dict(tuple(bars.sort_values("bar_index").groupby("date")))
    for S in fz.sessions:
        g = bg[S["sess"]]
        S["leg_dir"] = g["jma_leg_dir"].to_numpy(np.float64)
        jma = g["jma"].to_numpy(np.float64)
        tgt = g["is_target"].to_numpy()
        starts = np.where(tgt)[0]
        leg_id = np.cumsum(tgt)
        start_idx = np.concatenate(([0], starts))[leg_id]
        S["leg_start_jma"] = jma[start_idx]

In [6]:
# ---------------------------------------------------------------- grammar features
def build_grammar_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))
    return np.concatenate(blocks), fz.grammar_names

In [7]:
# ---------------------------------------------------------------- value features
def value_base_names(manifest):
    """Value channels derived from the manifest stream set (source columns)."""
    cols = manifest["_stream_cols"]
    names = ([f"z_{c}_signed" for c in cols]
             + [f"z_{c}_mag" for c in cols]
             + ["z_leg_amp", f"z_body_{BODY_TAG}_signed", f"z_body_{BODY_TAG}_mag"])
    return cols, names


def build_value_features(fz, src, date_from=None, date_to=None):
    cols, base = value_base_names(fz.manifest)
    names = base + [f"{nm}_lag{L}" for L in VALUE_LAGS for nm in base]
    sv = src.set_index("timestamp")
    blocks = []
    for S in fz._selected(date_from, date_to):
        ts = pd.DatetimeIndex(S["timestamp"])
        r = sv.reindex(ts)
        leg_dir = S["leg_dir"]
        feats = []
        for c in cols:                                             # signed per stream col
            feats.append(_expanding_z(r[c].to_numpy(np.float64) * leg_dir, ZWARM))
        for c in cols:                                             # magnitude per stream col
            feats.append(_expanding_z(np.abs(r[c].to_numpy(np.float64)), ZWARM))
        feats.append(_expanding_z(np.abs(r["JMA"].to_numpy(np.float64) - S["leg_start_jma"]), ZWARM))
        bo = r[BODY_OPEN_COL].to_numpy(np.float64)
        bc = r[BODY_CLOSE_COL].to_numpy(np.float64)
        feats.append(_expanding_z((bc - bo) * leg_dir, ZWARM))     # confirming-positive
        feats.append(_expanding_z(np.abs(bc - bo), ZWARM))

        M = np.stack(feats, 1)
        M = np.concatenate([np.zeros((1, M.shape[1])), M[:-1]], 0)          # t-1 shift
        lagged = [M]
        for L in VALUE_LAGS:
            lagged.append(np.concatenate([np.zeros((L, M.shape[1])), M[:-L]], 0))
        M = np.concatenate(lagged, 1)
        t = np.nonzero(~S["warm"])[0]
        Mt = M[t]
        Mt[(t < ZWARM + max(VALUE_LAGS))] = 0.0
        blocks.append(Mt.astype(np.float32))
    return np.concatenate(blocks), names


def build_X(fz, src, date_from=None, date_to=None):
    Xg, gn = build_grammar_features(fz, date_from, date_to)
    Xv, vn = build_value_features(fz, src, date_from, date_to)
    assert len(Xg) == len(Xv), (len(Xg), len(Xv))
    return np.hstack([Xg, Xv]), gn + vn


def build_meta(fz, date_from=None, date_to=None):
    bi, ts, tg, dt = [], [], [], []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        bi.append(S["bar_index"][t])
        ts.append(S["timestamp"][t])
        tg.append(S["tgt"][t])
        dt.append(np.full(len(t), str(S["sess"])))
    return pd.DataFrame({"bar_index": np.concatenate(bi),
                         "timestamp": np.concatenate(ts),
                         "is_target": np.concatenate(tg),
                         "date": np.concatenate(dt)})

In [8]:
# ---------------------------------------------------------------- train / eval
def train(fz, src, train_end, valid_from):
    X, names = build_X(fz, src, None, train_end)
    meta = build_meta(fz, None, train_end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    va = (meta["date"] >= valid_from).to_numpy()
    tr = ~va
    dtr = lgb.Dataset(X[tr], label=y[tr], feature_name=names)
    dva = lgb.Dataset(X[va], label=y[va], reference=dtr)
    booster = lgb.train(LGBM_PARAMS, dtr, num_boost_round=NUM_ROUNDS,
                        valid_sets=[dva], valid_names=["valid"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                                   lgb.log_evaluation(200)])
    p_va = booster.predict(X[va], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, y[va])
    print(json.dumps(dict(n_train=int(tr.sum()), n_valid=int(va.sum()),
                          best_iteration=int(booster.best_iteration),
                          valid_logloss_cal=float(log_loss(y[va], iso.predict(p_va)))),
                     indent=2))
    imp = pd.DataFrame({"feature": names,
                        "gain": booster.feature_importance("gain")}
                       ).sort_values("gain", ascending=False)
    print(imp.head(20).to_string(index=False))
    return dict(booster=booster, iso=iso, feature_names=names,
                valid_from=valid_from, train_end=train_end,
                manifest=fz.manifest, tag=RESULT_TAG, importance=imp)


def evaluate(fz, src, model, start, end=None, anchor=STAGE5_ANCHOR):
    X, _ = build_X(fz, src, start, end)
    meta = build_meta(fz, start, end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    p = model["booster"].predict(X, num_iteration=model["booster"].best_iteration)
    p_cal = model["iso"].predict(p)
    ll_cal = log_loss(y, p_cal)
    ll_const = log_loss(y, np.full_like(p, y.mean(), dtype=np.float64))
    print(json.dumps(dict(n_rows=int(len(y)), holdout_logloss_cal=float(ll_cal),
                          holdout_logloss_const=float(ll_const),
                          skill=float(1 - ll_cal / ll_const),
                          auc=float(roc_auc_score(y, p)),
                          anchor=anchor, delta=float(ll_cal - anchor)), indent=2))
    out = meta[["bar_index", "timestamp", "is_target"]].copy()
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)
    tbl = out.assign(bin=pd.qcut(out["p_cal"], 10, duplicates="drop")).groupby(
        "bin", observed=True).agg(mean_p=("p_cal", "mean"),
                                  realized=("is_target", "mean"), n=("p_cal", "size"))
    print(tbl.to_string())
    return out

In [9]:
# ---------------------------------------------------------------- run
manifest = load_manifest(MANIFEST_PATH)

SOURCE_PATH = manifest["source_file"]
RAW_FILE = manifest["raw_file"]

assert STAGE0_TAG == manifest["stage0_tag"]

print('------------ !! VERIFY !! ------------')
print(f'FRAME: {FRAME}sec, STAGE0_TAG: {STAGE0_TAG}')
print('-------------- MANIFEST --------------')
print(json.dumps({k: v for k, v in manifest.items() if not k.startswith("_")}, indent=2))

bars = pd.read_parquet(BARS_PATH)
events = pd.read_parquet(EVENTS_PATH)

sess_lo = pd.Timestamp(manifest["session_start"]).time()
sess_hi = pd.Timestamp(manifest["session_end"]).time()

src = pd.read_parquet(SOURCE_PATH)
src = src[(src["timestamp"].dt.time >= sess_lo) & (src["timestamp"].dt.time < sess_hi)]

raw1 = pd.read_parquet(RAW_FILE)
raw1 = raw1[(raw1["timestamp"].dt.time >= sess_lo) & (raw1["timestamp"].dt.time < sess_hi)]
raw1 = raw1.rename(columns={"Open": "rawOpen", "High": "rawHigh", "Low": "rawLow", "Last": "rawLast"})

src = src.merge(raw1[["timestamp", "rawOpen", "rawHigh", "rawLow", "rawLast"]], on="timestamp", how="left")

assert src[["rawOpen", "rawLast"]].notna().all().all(), "raw OHLC has gaps vs source timestamps"

------------ !! VERIFY !! ------------
FRAME: 3sec, STAGE0_TAG: NOTICK
-------------- MANIFEST --------------
{
  "frame_seconds": 3,
  "session_start": "09:30",
  "session_end": "16:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "../data/mnq-tick-oscillator-3sec.pqt",
  "raw_file": "../data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 8916187,
  "n_sessions": 1163,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-13T16:12:55.299307",
  "stage0_tag": "NOTICK"
}


In [10]:
fz = Featurizer(bars, events, manifest)
augment_featurizer(fz, bars)

S0 = fz.sessions[len(fz.sessions) // 2]
xchk = src.set_index("timestamp").reindex(pd.DatetimeIndex(S0["timestamp"]))["jmaD1"].to_numpy(np.float64)
print("welford max abs diff:", _welford_check(xchk, ZWARM))

model = train(fz, src, TRAIN_END, VALID_FROM)
pred = evaluate(fz, src, model, TEST_FROM)

print(f"-------------- {RESULT_TAG} {FRAME}s --------------")
joblib.dump({k: v for k, v in model.items() if k != "importance"},
            f"{OUT_DIR}/lgbm5_model_{RESULT_TAG}_{FRAME}s.joblib")
model["importance"].to_csv(f"{OUT_DIR}/lgbm5_importance_{RESULT_TAG}_{FRAME}s.csv", index=False)
pred.to_parquet(f"{OUT_DIR}/pred_lgbm5_{RESULT_TAG}_{FRAME}s.pqt", index=False)

z = pred.timestamp >= TEST_FROM
print("holdout-window logloss:", log_loss(pred.is_target[z], pred.p_cal[z]))

welford max abs diff: 2.220446049250313e-14
[200]	valid's binary_logloss: 0.138117
[400]	valid's binary_logloss: 0.133349
[600]	valid's binary_logloss: 0.131463
[800]	valid's binary_logloss: 0.130574
[1000]	valid's binary_logloss: 0.130034
[1200]	valid's binary_logloss: 0.129718
[1400]	valid's binary_logloss: 0.129489
[1600]	valid's binary_logloss: 0.129304
[1800]	valid's binary_logloss: 0.129157
[2000]	valid's binary_logloss: 0.129042
[2200]	valid's binary_logloss: 0.128962
[2400]	valid's binary_logloss: 0.128913
[2600]	valid's binary_logloss: 0.128866
[2800]	valid's binary_logloss: 0.128845
[3000]	valid's binary_logloss: 0.128827
[3200]	valid's binary_logloss: 0.128815
[3400]	valid's binary_logloss: 0.128783
[3600]	valid's binary_logloss: 0.12877
[3800]	valid's binary_logloss: 0.128753
[4000]	valid's binary_logloss: 0.128743
{
  "n_train": 6892422,
  "n_valid": 986412,
  "best_iteration": 3999,
  "valid_logloss_cal": 0.12841370374285066
}
                feature         gain
      z_